<div style="font-size: 1em; display: flex; align-items: center; gap: 8px; padding: 8px 16px; background: #F8F9FA; border-bottom: 2px solid #E0E0E0; margin: 0; line-height: 1">
    <img src="https://cdn.simpleicons.org/databricks/FF3621" width="24" height="24"/>
    <div style="color: #666">
        <span style="font-weight: bold; color: #333">Databricks Performance Optimization
</span>
        <span style="margin-left: 8px; color: #999">|</span>
        <span style="margin-left: 8px">09 Lecture - Serverless and Photon</span>
    </div>
</div>

![DB Academy](https://files.training.databricks.com/binder/prod_main/databricks-performance-optimization-en_us-3.0.1/images/20261106T135523Z/Databricks Performance Optimization/Includes/Images/common/databricks_academy_logo.png)

# Lecture - Serverless First and Photon
#### Duration: 15 minutes

## Overview

Parts 2 through 8 fixed layout, workload patterns, and code. This part covers the compute layer, where the default is serverless.

In this part we cover:
- the 4 serverless surfaces
- the 3 hard prerequisites that decide whether a job can run serverless at all
- the short list of cases where classic compute is still recommended

We also cover the cost model: how serverless is metered and the performance modes that trade startup latency for a lower bill.

We then turn to **Photon**, the vectorized engine underneath both serverless and classic, and how to confirm it actually engaged.



## Learning Objectives

By the end of this part, you will be able to:

1. **Name** the 4 serverless surfaces: Jobs, Notebooks, DBSQL warehouses and SDP.
2. **State** the 3 hard serverless prerequisites: Standard access mode, Unity Catalog only reads and writes and Spark Connect compatible code.
3. **Identify** the legitimate classic only exceptions and explain how Workspace Base Environments replace dependency init scripts.
4. **Distinguish** performance optimized from standard mode for serverless Jobs and reason about the cost and latency trade.
5. **Verify** that Photon engaged from the Query Profile: percent of task time in Photon and the COVERAGE_PHOTON insight.

## A. The 4 Serverless Surfaces
<br/>
<div class="cw">
<style>
.cw { font-family: sans-serif; max-width: 1100px; margin: 0 auto; color: #0b2026; display: flex; flex-direction: column; gap: 20px; }
.cw * { box-sizing: border-box; }
.ds-banner { display: flex; align-items: center; gap: 20px; width: 100%; border-radius: 10px; padding: 24px 28px; background: var(--bnr-bg); border-left: 8px solid var(--bnr-accent); box-shadow: 0 2px 10px rgba(27,49,57,0.08); }
.ds-banner-icon { width: 60px; height: 60px; border-radius: 50%; flex-shrink: 0; background: var(--bnr-accent); color: var(--bnr-icon-fg); font-size: 30pt; font-weight: 700; line-height: 1; display: flex; align-items: center; justify-content: center; }
.ds-banner-title { font-size: 22pt; font-weight: 700; color: var(--bnr-title); line-height: 1.2; margin-bottom: 4px; }
.ds-banner-tagline { font-size: 15pt; color: var(--bnr-tagline); line-height: 1.4; }
.ds-banner.info { --bnr-bg: #1B5162; --bnr-accent: #4299E0; --bnr-icon-fg: #FFFFFF; --bnr-title: #FFFFFF; --bnr-tagline: #F9F7F4; }
.ds-row { display: flex; gap: 24px; align-items: stretch; flex-wrap: wrap; }
.ds-row > * { flex: 1; min-width: 240px; }
.ds-card { background: #F9F7F4; border-radius: 8px; box-shadow: 0 2px 8px rgba(27,49,57,0.06); padding: 20px; border-top: 8px solid var(--accent, #4299E0); }
.blue{--accent:#4299E0;} .green{--accent:#00A972;} .coral{--accent:#FF5F46;} .amber{--accent:#FFAB00;} .red{--accent:#98102A;}
.ds-card-title { font-size: 18pt; font-weight: 700; line-height: 1.25; margin: 0 0 8px; }
.ds-card-text { font-size: 14pt; color: #5E7077; line-height: 1.55; margin: 0; }
.tc { text-align: center; }
</style>

<div class="ds-banner info">
  <div class="ds-banner-icon">i</div>
  <div>
    <div class="ds-banner-title">Serverless is the default compute for all 4 surfaces</div>
    <div class="ds-banner-tagline">You do not pick node types, set autotermination values, or pay for idle capacity. Billing is per second of actual compute.</div>
  </div>
</div>

<div class="ds-row">
  <div class="ds-card blue tc"><div class="ds-card-title">Jobs</div><div class="ds-card-text">Scheduled and triggered <strong>Lakeflow Jobs</strong> run on serverless job compute. The cluster starts on demand and stops at the end.</div></div>
  <div class="ds-card green tc"><div class="ds-card-title">Notebooks</div><div class="ds-card-text">Interactive <strong>notebooks</strong> attach to serverless in seconds. This notebook runs on it.</div></div>
  <div class="ds-card coral tc"><div class="ds-card-title">DBSQL Warehouses</div><div class="ds-card-text">Serverless <strong>SQL warehouses</strong> back the SQL editor, dashboards, and BI tools with fast start and elastic scale.</div></div>
  <div class="ds-card amber tc"><div class="ds-card-title">SDP</div><div class="ds-card-text"><strong>Lakeflow Spark Declarative Pipelines</strong> run serverless with enhanced autoscaling built in.</div></div>
</div>
</div>

##### EXPAND FOR ADDITIONAL DETAILS

<details>

### Why one compute model across four surfaces matters
- The same serverless plane backs Jobs, Notebooks, DBSQL warehouses and SDP.
- What disappears on all four: choosing node types, sizing workers, setting an autotermination timeout, and paying for idle capacity while a cluster spins up or winds down.
- What you pay for: the seconds of compute actually used. Start is measured in seconds, not the minutes a classic cluster takes to provision VMs.

### How this maps to the rest of the course
- Part 2 measured queries on a serverless notebook and a serverless SQL warehouse. Those are two of these four surfaces.
- Section C below covers when you still reach for a classic cluster. Those are the documented exceptions.

</details>

### A1. The Three Hard Prerequisites
<br/>
<div class="gatex">
  <style>
    .gatex { font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif; color: #1f2933; background: #ffffff; padding: 18px 16px 22px 16px; border: 1px solid #e2e8ee; border-radius: 10px; max-width: 1040px; margin: 0 auto; }
    .gatex h3 { margin: 0 0 4px 0; font-size: 19px; font-weight: 650; color: #14343f; letter-spacing: .1px; }
    .gatex p.sub { margin: 0 0 14px 0; font-size: 15px; color: #5f6b76; }
    .gatex svg { display: block; width: 100%; height: auto; }
    .gatex text { font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif; }
    .gatex .gnum { font-size: 15px; font-weight: 700; fill: #ffffff; }
    .gatex .gttl { font-size: 14px; font-weight: 700; fill: #14343f; }
    .gatex .sm { font-size: 12px; fill: #62707d; }
    .gatex .tag { font-size: 11px; font-weight: 700; letter-spacing: .4px; }
    .gatex .note { margin-top: 12px; font-size: 14px; color: #5f6b76; border-left: 3px solid #0f9aab; padding: 4px 0 4px 10px; line-height: 1.55; }
    .gatex code { font-family: ui-monospace, SFMono-Regular, Menlo, Consolas, monospace; font-size: 14px; background: #f2f5f8; padding: 1px 4px; border-radius: 3px; color: #23424c; }
  </style>
  <h3>Three Gates a Job Must Pass to Run on Serverless</h3>
  <p class="sub">These are platform level enforcement. Trip any one and the job will not run on serverless.</p>
  <svg viewBox="0 0 1040 276" role="img" aria-label="A job passes through three sequential gates: Standard access mode, Unity Catalog only, and Spark Connect compatible code. Passing all three reaches serverless.">
    <defs>
      <marker id="gatex-arr" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse">
        <path d="M 0 0 L 10 5 L 0 10 z" fill="#0f9aab"></path>
      </marker>
    </defs>
    <!-- your job -->
    <rect x="16" y="70" width="120" height="90" rx="8" fill="#f7fafc" stroke="#b9c7d2"></rect>
    <text class="gttl" x="76" y="108" text-anchor="middle">Your job</text>
    <text class="sm" x="76" y="128" text-anchor="middle">wants to run</text>
    <text class="sm" x="76" y="144" text-anchor="middle">serverless</text>
    <line x1="136" y1="115" x2="166" y2="115" stroke="#0f9aab" stroke-width="1.8" marker-end="url(#gatex-arr)"></line>
    <!-- gate 1 -->
    <rect x="168" y="56" width="228" height="118" rx="8" fill="#eef6fd" stroke="#9fcdd6"></rect>
    <circle cx="192" cy="80" r="13" fill="#4299E0"></circle>
    <text class="gnum" x="192" y="85" text-anchor="middle">1</text>
    <text class="gttl" x="214" y="85">Standard access mode</text>
    <text class="sm" x="184" y="112">Formerly "shared". Enforces UC</text>
    <text class="sm" x="184" y="130">fine-grained access via Lakeguard.</text>
    <text class="sm" x="184" y="148">Row filters, masks, dynamic views.</text>
    <line x1="396" y1="115" x2="426" y2="115" stroke="#0f9aab" stroke-width="1.8" marker-end="url(#gatex-arr)"></line>
    <!-- gate 2 -->
    <rect x="428" y="56" width="228" height="118" rx="8" fill="#eef6fd" stroke="#9fcdd6"></rect>
    <circle cx="452" cy="80" r="13" fill="#4299E0"></circle>
    <text class="gnum" x="452" y="85" text-anchor="middle">2</text>
    <text class="gttl" x="474" y="85">Unity Catalog only</text>
    <text class="sm" x="444" y="112">Reads and writes go through UC.</text>
    <text class="sm" x="444" y="130">No mounts, no <tspan font-family="monospace">dbfs:/</tspan>, no</text>
    <text class="sm" x="444" y="148">hive_metastore.</text>
    <line x1="656" y1="115" x2="686" y2="115" stroke="#0f9aab" stroke-width="1.8" marker-end="url(#gatex-arr)"></line>
    <!-- gate 3 -->
    <rect x="688" y="56" width="228" height="118" rx="8" fill="#eef6fd" stroke="#9fcdd6"></rect>
    <circle cx="712" cy="80" r="13" fill="#4299E0"></circle>
    <text class="gnum" x="712" y="85" text-anchor="middle">3</text>
    <text class="gttl" x="734" y="85">Spark Connect ready</text>
    <text class="sm" x="704" y="112">DataFrame and SQL APIs only.</text>
    <text class="sm" x="704" y="130">No <tspan font-family="monospace">sparkContext</tspan>, no RDDs,</text>
    <text class="sm" x="704" y="148">no Py4J JVM access.</text>
    <line x1="916" y1="115" x2="946" y2="115" stroke="#00A972" stroke-width="2" marker-end="url(#gatex-arr)"></line>
    <!-- serverless (pass) -->
    <rect x="948" y="56" width="80" height="118" rx="8" fill="#00A972"></rect>
    <text class="gnum" x="988" y="104" text-anchor="middle">Runs</text>
    <text class="sm" style="fill:#eafaf1" x="988" y="124" text-anchor="middle">on</text>
    <text class="sm" style="fill:#eafaf1" x="988" y="140" text-anchor="middle">serverless</text>
    <!-- fail note -->
    <rect x="168" y="196" width="748" height="70" rx="6" fill="#fdf2f4" stroke="#d4808f"></rect>
    <text x="184" y="220" style="font-size:20px;fill:#98102A;font-family:-apple-system,BlinkMacSystemFont,Segoe UI,Roboto,Helvetica,Arial,sans-serif">Trip any gate and the job is rejected.</text>
    <text x="184" y="250" style="font-size:20px;fill:#98102A;font-family:-apple-system,BlinkMacSystemFont,Segoe UI,Roboto,Helvetica,Arial,sans-serif">Check your code against all three before moving a job to serverless.</text>
  </svg>
  <div class="note">The access-mode names changed: "shared" is now <strong>Standard</strong>, "single-user" is now <strong>Dedicated</strong>. Standard is the multi-user target for serverless, and Lakeguard is what isolates user code so row filters, column masks, and dynamic views are safe to apply. Most modern DataFrame and SQL code already clears gate 3. Code that drops to the RDD API or reaches into <code>_jvm</code> is the part that needs rewriting.</div>
</div>

##### EXPAND FOR ADDITIONAL DETAILS

<details>

### Standard access mode
- The access mode names changed. "Shared" is now **Standard**, and "single-user" is now **Dedicated**. Standard is the multi user target for serverless.
- Standard enforces Unity Catalog fine grained access control natively, using **Lakeguard** to isolate user code from other users and from the host. That isolation is why row filters, column masks, and dynamic views are safe to apply.
- Dedicated compute can also honor fine grained access control, but only when the workspace has serverless enabled and runs DBR 15.4 LTS or later, because it delegates the filtering to serverless.

### Unity Catalog only
- Serverless reads and writes go through Unity Catalog. Mounts, raw DBFS paths, and the legacy `hive_metastore` are not available.
- In practice this means migrating external or mounted data into Unity Catalog and referencing tables by their `catalog.schema.table` name.

### Spark Connect compatible code
- Serverless uses Spark Connect, which decouples your client from the Spark driver. That decoupling is what forbids `sparkContext`, RDDs, and any Py4J call into the JVM.
- The DataFrame and SQL APIs are Spark Connect compatible, so most modern code already qualifies. Code that drops to the RDD API or reaches into `_jvm` is the part that needs rewriting.
- This is a de-teach point too. RDD level tuning and `sc.broadcast` are legacy. The DataFrame API plus AQE and Photon replace them.

</details>

## B. When Classic Still Wins
<br/>
<div class="cw">
<style>
.cw { font-family: sans-serif; max-width: 1100px; margin: 0 auto; color: #0b2026; display: flex; flex-direction: column; gap: 20px; }
.cw * { box-sizing: border-box; }
.ds-banner { display: flex; align-items: center; gap: 20px; width: 100%; border-radius: 10px; padding: 24px 28px; background: var(--bnr-bg); border-left: 8px solid var(--bnr-accent); box-shadow: 0 2px 10px rgba(27,49,57,0.08); }
.ds-banner-icon { width: 60px; height: 60px; border-radius: 50%; flex-shrink: 0; background: var(--bnr-accent); color: var(--bnr-icon-fg); font-size: 30pt; font-weight: 700; line-height: 1; display: flex; align-items: center; justify-content: center; }
.ds-banner-title { font-size: 22pt; font-weight: 700; color: var(--bnr-title); line-height: 1.2; margin-bottom: 4px; }
.ds-banner-tagline { font-size: 15pt; color: var(--bnr-tagline); line-height: 1.4; }
.ds-banner.caution { --bnr-bg: rgba(255,171,0,0.12); --bnr-accent: #FFAB00; --bnr-icon-fg: #0b2026; --bnr-title: #0b2026; --bnr-tagline: #5E7077; }
.ds-row { display: flex; gap: 24px; align-items: stretch; flex-wrap: wrap; }
.ds-row > * { flex: 1; min-width: 240px; }
.ds-card { background: #F9F7F4; border-radius: 8px; box-shadow: 0 2px 8px rgba(27,49,57,0.06); padding: 20px; border-top: 8px solid var(--accent, #4299E0); }
.blue{--accent:#4299E0;} .green{--accent:#00A972;} .coral{--accent:#FF5F46;} .amber{--accent:#FFAB00;} .red{--accent:#98102A;}
.ds-card-title { font-size: 18pt; font-weight: 700; line-height: 1.25; margin: 0 0 8px; }
.ds-card ul { margin: 8px 0 0; padding-left: 20px; font-size: 14pt; line-height: 1.55; color: #0b2026; }
.ds-card li { margin-bottom: 10px; } .ds-card li:last-child { margin-bottom: 0; }
.ds-callout { border-left: 6px solid #00A972; border-radius: 8px; padding: 16px 20px; background: rgba(0,169,114,0.10); font-size: 14pt; line-height: 1.55; color: #0b2026; }
.ds-eyebrow { font-size: 14pt; font-weight: 700; letter-spacing: 1.5px; text-transform: uppercase; color: #00A972; margin-bottom: 6px; }
</style>

<div class="ds-banner caution">
  <div class="ds-banner-icon">!</div>
  <div>
    <div class="ds-banner-title">The documented exception list</div>
    <div class="ds-banner-tagline">These are the specific capabilities serverless does not cover today. Each one is a documented exception.</div>
  </div>
</div>

<div class="rc-wrap">
<style>
.rc-wrap { width: 100%; display: flex; flex-direction: column; align-items: center; gap: 18px; font-family: sans-serif; }
.rc-wrap * { box-sizing: border-box; }
.rc-hint { width: 100%; max-width: 1100px; text-align: center; font-size: 14pt; color: #5E7077; font-style: italic; }
.rc-hint strong { color: #FF5F46; font-style: normal; font-weight: 700; }
.rc-row { display: block; white-space: nowrap; font-size: 0; width: 100%; max-width: 1100px; }
.rc-box { display: inline-block; width: 32%; margin-right: 2%; min-height: 92px; background: #F9F7F4; border-top: 8px solid transparent; border-left: 2px solid transparent; border-right: 2px solid transparent; border-bottom: 2px solid transparent; border-radius: 8px; padding: 16px 14px; text-align: center; cursor: pointer; user-select: none; vertical-align: top; transition: transform 0.12s; box-shadow: 0 2px 8px rgba(27,49,57,0.06); white-space: normal; }
.rc-box:last-child { margin-right: 0; }
.rc-box:hover { transform: translateY(-2px); }
.rc-box.active { background: #fff; border-left-color: var(--pc); border-right-color: var(--pc); border-bottom-color: var(--pc); }
.rc-label { display: block; font-weight: 700; font-size: 14pt; letter-spacing: 1px; text-transform: uppercase; line-height: 1.2; pointer-events: none; }
.rc-title { display: block; font-size: 14pt; color: #0b2026; line-height: 1.3; margin-top: 6px; pointer-events: none; }
.rc-detail-wrap { overflow: hidden; max-height: 0; opacity: 0; transition: max-height 0.35s ease, opacity 0.28s ease, margin-top 0.28s ease; margin-top: 0; width: 100%; max-width: 1100px; }
.rc-detail-wrap.open { max-height: 700px; opacity: 1; margin-top: 12px; }
.rc-detail-card { background: #F9F7F4; border-radius: 10px; padding: 22px 26px; border-top: 7px solid #ccc; box-shadow: 0 2px 8px rgba(27,49,57,0.06); }
.rc-detail-title { font-size: 17pt; font-weight: 700; color: #0b2026; margin-bottom: 12px; line-height: 1.3; }
.rc-detail-body { font-size: 14pt; color: #0b2026; line-height: 1.55; }
.rc-detail-body ul { margin: 0; padding-left: 22px; }
.rc-detail-body li { margin-bottom: 8px; }
.rc-detail-body li:last-child { margin-bottom: 0; }
</style>

<div class="rc-hint">Three kinds of workload that stay on classic. <strong>Click each</strong> for the specifics.</div>

<div class="rc-row">
  <span class="rc-box" data-id="0" onclick="rcSelect(0)" style="--pc:#98102A; border-top-color:#98102A;">
    <span class="rc-label" style="color:#98102A;">Language and runtime</span>
    <span class="rc-title">Scala, R, unsupported runtimes</span>
  </span><span class="rc-box" data-id="1" onclick="rcSelect(1)" style="--pc:#FFAB00; border-top-color:#FFAB00;">
    <span class="rc-label" style="color:#B8860B;">Node and network</span>
    <span class="rc-title">GPU, private subnet, proven tuning</span>
  </span><span class="rc-box" data-id="2" onclick="rcSelect(2)" style="--pc:#4299E0; border-top-color:#4299E0;">
    <span class="rc-label" style="color:#4299E0;">Environment setup</span>
    <span class="rc-title">Init scripts, OS-level packages</span>
  </span>
</div>

<div class="rc-detail-wrap" id="rc-detail-wrap">
  <div class="rc-detail-card" id="rc-detail-card"></div>
</div>
</div>

<script>
var REVEAL = [
  { title: "Language and runtime", color: "#98102A", text: "<ul><li><strong>Scala</strong> notebooks and <strong>R</strong> workloads. These are the hard no for serverless notebooks right now.</li><li>Libraries that need a specific unsupported runtime.</li></ul>" },
  { title: "Node and network", color: "#FFAB00", text: "<ul><li><strong>GPU</strong> instances and <strong>private subnet</strong> networking with strict egress control, infrastructure serverless does not expose.</li><li>Genuinely knob-dependent Spark tuning you can prove you need. Most tuning is now automatic (AQE, Photon, Predictive Optimization).</li></ul>" },
  { title: "Environment setup", color: "#4299E0", text: "<ul><li>Custom <strong>init scripts</strong> or OS-level packages.</li><li>Being modernized: <strong>Workspace Base Environments</strong> now cover Python dependencies on serverless.</li><li>An init-script migration path is on the roadmap.</li></ul>" }
];
var rcCurrent = null;
function rcSelect(id) {
  var wrap = document.getElementById('rc-detail-wrap');
  var card = document.getElementById('rc-detail-card');
  var d = REVEAL[id];
  document.querySelectorAll('.rc-box').forEach(function(b) {
    b.classList.toggle('active', parseInt(b.dataset.id) === id && rcCurrent !== id);
  });
  if (rcCurrent === id) { wrap.classList.remove('open'); rcCurrent = null; return; }
  rcCurrent = id;
  card.style.borderTopColor = d.color;
  card.innerHTML = '<div class="rc-detail-title" style="color:' + d.color + ';">' + d.title + '</div><div class="rc-detail-body">' + d.text + '</div>';
  wrap.classList.add('open');
}
</script>

<div class="ds-callout">
  <strong>Init scripts for Python dependencies are legacy on serverless.</strong>
  <ul style="margin:8px 0 0;padding-left:20px;">
    <li><strong>Workspace Base Environments (WBE)</strong> replace them: a declarative YAML listing an environment version plus Python packages. An admin builds and caches it, sessions start with the right packages already present.</li>
    <li>Init scripts remain only for <strong>OS-level or kernel-level</strong> setup (apt packages, kernel tweaks). Needing one is itself a reason the workload stays on classic.</li>
  </ul>
</div>
</div>

##### EXPAND FOR ADDITIONAL DETAILS

<details>

### Read this as a finite exception list
- Each item is a specific capability serverless does not offer today. Read the list as a finite exception set, not a general argument for classic.
- Scala and R are the hard "no" for serverless notebooks right now. If a workload is Scala or R, it stays on classic.
- GPU and private subnet networking are infrastructure needs serverless does not expose.
- "Genuinely knob dependent tuning" is deliberately narrow. Most tuning people think they need is now automatic (AQE, Photon, Predictive Optimization). Reach for classic only when you can prove a specific config changes the outcome.

### Workspace Base Environments replace dependency init scripts
- Serverless has no init scripts and no cluster policies in the classic sense. Historically teams used init scripts to `pip install` dependencies. On serverless that job belongs to Workspace Base Environments.
- A WBE is a declarative YAML: a serverless environment version plus a list of Python packages. An admin builds and caches it, and notebooks or jobs select it, so sessions start fast with the right packages already present.
- WBE covers Python dependency management. It does not cover OS level changes (apt packages, kernel tweaks). Those still need an init script, and needing one is itself a reason the workload stays on classic.
- The init script migration agent is on the roadmap, which will shrink this exception further. Verify its status before you promise it.

### What serverless retires
- `spark.dynamicAllocation.*`, executor and GC sizing, "5 cores per executor" math. Serverless owns resources.
- Instance pools (existed to hide VM boot time). Serverless starts in seconds.
- Hand-tuned JVM internals. Replaced by Photon (C++ engine) and Arrow (Python paths).

</details>

## C. Cost Model and Performance Modes
<br/>
<div class="cw">
<style>
.cw { font-family: sans-serif; max-width: 1100px; margin: 0 auto; color: #0b2026; display: flex; flex-direction: column; gap: 20px; }
.cw * { box-sizing: border-box; }
.fc-row { display: flex; gap: 24px; align-items: stretch; flex-wrap: wrap; }
.fc-card { background: #F9F7F4; border-radius: 10px; padding: 20px; border-top: 8px solid var(--accent); box-shadow: 0 2px 8px rgba(27,49,57,0.06); flex: 1; min-width: 300px; }
.fc-title { font-size: 18pt; font-weight: 700; color: #0b2026; margin-bottom: 14px; }
.fc-intro { font-size: 14pt; line-height: 1.55; color: #0b2026; margin-bottom: 16px; }
.fc-list { margin-bottom: 18px; }
.fc-item { display: grid; grid-template-columns: 150px 1fr; gap: 20px; padding: 12px 0; border-bottom: 1px solid #EEEDE9; align-items: start; }
.fc-item:last-child { border-bottom: none; padding-bottom: 4px; }
.fc-key { font-size: 14pt; font-weight: 700; letter-spacing: 1px; text-transform: uppercase; line-height: 1.3; color: var(--accent); }
.fc-val { font-size: 14pt; color: #0b2026; line-height: 1.55; }
.fc-example { border-left: 4px solid var(--accent); border-radius: 0 8px 8px 0; padding: 14px 16px; font-size: 14pt; line-height: 1.55; background: var(--tint); }
.fc-example strong:first-child { text-transform: uppercase; font-size: 14pt; letter-spacing: 1.5px; display: block; margin-bottom: 6px; }
.ds-callout { border-left: 6px solid #FFAB00; border-radius: 8px; padding: 16px 20px; background: rgba(255,171,0,0.12); font-size: 14pt; line-height: 1.55; color: #0b2026; }
</style>

<div style="font-size:15pt;line-height:1.55;">Serverless is metered per workload second, with no separate VM bill and no idle capacity to pay for. Serverless <strong>Jobs</strong> and <strong>pipelines</strong> add one lever worth knowing: the performance mode.</div>

<div class="fc-row">
<div class="fc-card" style="--accent:#4299E0; --tint:rgba(66,153,224,0.10);">
  <div class="fc-title">Performance optimized</div>
  <div class="fc-intro">Keeps resources warm for the fastest start.</div>
  <div class="fc-list">
    <div class="fc-item"><div class="fc-key">Start</div><div class="fc-val">Seconds.</div></div>
    <div class="fc-item"><div class="fc-key">Scaling</div><div class="fc-val">Aggressive.</div></div>
    <div class="fc-item"><div class="fc-key">Best for</div><div class="fc-val">Interactive and latency sensitive runs.</div></div>
  </div>
  <div class="fc-example"><strong>Use it when</strong>Someone or something is waiting on the result right now.</div>
</div>
<div class="fc-card" style="--accent:#00A972; --tint:rgba(0,169,114,0.10);">
  <div class="fc-title">Standard</div>
  <div class="fc-intro">Trades startup latency for a lower bill.</div>
  <div class="fc-list">
    <div class="fc-item"><div class="fc-key">Start</div><div class="fc-val">Higher startup latency.</div></div>
    <div class="fc-item"><div class="fc-key">Scaling</div><div class="fc-val">Conservative.</div></div>
    <div class="fc-item"><div class="fc-key">Best for</div><div class="fc-val">Scheduled batch Jobs and pipelines.</div></div>
  </div>
  <div class="fc-example"><strong>Use it when</strong>A nightly batch job can absorb a slower start for a meaningfully lower bill.</div>
</div>
</div>

<div class="ds-callout">
  <strong>Standard mode is for Jobs and pipelines, not notebooks.</strong> Interactive notebooks always run performance optimized, which is why this notebook cannot toggle the mode. You set it on a serverless Job or pipeline.
</div>
</div>

##### EXPAND FOR ADDITIONAL DETAILS

<details>

### How serverless is metered
- You pay for the compute seconds actually used, billed in DBUs, with no separate VM line item and no idle capacity charge.
- Classic splits the bill into VM cost plus DBUs and keeps billing while the cluster is up, even when idle. On serverless, you pay only for the seconds of actual work.

### Performance optimized versus standard
- Performance optimized keeps resources warm so a run starts in seconds and scales aggressively. It is the default and the only option for interactive notebooks.
- Standard mode accepts a higher startup latency and scales conservatively, in exchange for a lower bill on the same work. Databricks documents savings up to around 70 percent versus performance optimized for suitable workloads.
- Standard mode is available for Lakeflow Jobs and Lakeflow pipelines, not for notebooks. You choose it in the Job or pipeline settings.
- Verify exact savings in the docs and your own account. The number depends on the workload.

### Which mode actually ran
- For a serverless Job run, the Jobs API `runs/get` response carries the `performance_target`. You can also read `product_features.performance_target` in `system_vw.billing.usage` to compare cost by mode across many runs.
- Report DBUs per run by mode to confirm a batch job is getting the standard mode discount.

### Usage policies and attribution
- Serverless spend is governed with usage policies and budgets, and attribution relies on tags. Untagged serverless usage is unattributable, so tag everything you want to charge back. Part 6 goes deeper on cost governance.
- ANSI SQL mode is on by default on serverless (and on DBR 17.3). Migrating a permissive classic job can surface new errors on overflow or bad casts.

</details>

## D. Photon, the Vectorized Engine
<br/>
<div class="cw">
<style>
.cw { font-family: sans-serif; max-width: 1100px; margin: 0 auto; color: #0b2026; display: flex; flex-direction: column; gap: 20px; }
.cw * { box-sizing: border-box; }
.ds-banner { display: flex; align-items: center; gap: 20px; width: 100%; border-radius: 10px; padding: 24px 28px; background: var(--bnr-bg); border-left: 8px solid var(--bnr-accent); box-shadow: 0 2px 10px rgba(27,49,57,0.08); }
.ds-banner-icon { width: 60px; height: 60px; border-radius: 50%; flex-shrink: 0; background: var(--bnr-accent); color: var(--bnr-icon-fg); font-size: 30pt; font-weight: 700; line-height: 1; display: flex; align-items: center; justify-content: center; }
.ds-banner-title { font-size: 22pt; font-weight: 700; color: var(--bnr-title); line-height: 1.2; margin-bottom: 4px; }
.ds-banner-tagline { font-size: 15pt; color: var(--bnr-tagline); line-height: 1.4; }
.ds-banner.info { --bnr-bg: #1B5162; --bnr-accent: #4299E0; --bnr-icon-fg: #FFFFFF; --bnr-title: #FFFFFF; --bnr-tagline: #F9F7F4; }
.ds-row { display: flex; gap: 24px; align-items: stretch; flex-wrap: wrap; }
.ds-row > * { flex: 1; min-width: 240px; }
.ds-card { background: #F9F7F4; border-radius: 8px; box-shadow: 0 2px 8px rgba(27,49,57,0.06); padding: 20px; border-top: 8px solid var(--accent, #4299E0); }
.blue{--accent:#4299E0;} .green{--accent:#00A972;} .coral{--accent:#FF5F46;} .amber{--accent:#FFAB00;} .red{--accent:#98102A;}
.ds-card-title { font-size: 18pt; font-weight: 700; line-height: 1.25; margin: 0 0 8px; }
.ds-card ul { margin: 8px 0 0; padding-left: 20px; font-size: 14pt; line-height: 1.55; color: #0b2026; }
.ds-card li { margin-bottom: 10px; } .ds-card li:last-child { margin-bottom: 0; }
.ds-callout { border-left: 6px solid #4299E0; border-radius: 8px; padding: 16px 20px; background: #F8F9FC; font-size: 14pt; line-height: 1.55; color: #0b2026; }
.ds-eyebrow { font-size: 14pt; font-weight: 700; letter-spacing: 1.5px; text-transform: uppercase; color: #4299E0; margin-bottom: 6px; }
</style>

<div class="ds-banner info">
  <div class="ds-banner-icon">i</div>
  <div>
    <div class="ds-banner-title">A vectorized C++ engine under Spark</div>
    <div class="ds-banner-tagline">Default on for serverless, opt in per cluster on classic. Your APIs and optimizer stay the same, only the execution layer changes.</div>
  </div>
</div>

<div style="background:#ffffff;border:1px solid #e2e8ee;border-radius:10px;padding:16px;">
  <svg viewBox="0 0 1040 282" role="img" aria-label="The JVM engine processes one row at a time. Photon processes a columnar batch of values at once, which is why it is faster." style="display:block;width:100%;height:auto;font-family:-apple-system,BlinkMacSystemFont,sans-serif;">
    <!-- left: row-at-a-time JVM -->
    <text x="20" y="28" font-size="14" font-weight="700" fill="#14343f">JVM engine: one row at a time</text>
    <text x="20" y="46" font-size="12" fill="#62707d">each row processed on its own, with per-row overhead</text>
    <g>
      <rect x="20" y="60" width="200" height="24" rx="4" fill="#eef1f4" stroke="#c6d2db"></rect>
      <rect x="20" y="60" width="40" height="24" rx="4" fill="#8a97a3"></rect>
      <text x="230" y="77" font-size="12" fill="#62707d">row 1</text>
      <rect x="20" y="92" width="200" height="24" rx="4" fill="#eef1f4" stroke="#c6d2db"></rect>
      <rect x="20" y="92" width="40" height="24" rx="4" fill="#8a97a3"></rect>
      <text x="230" y="109" font-size="12" fill="#62707d">row 2</text>
      <rect x="20" y="124" width="200" height="24" rx="4" fill="#eef1f4" stroke="#c6d2db"></rect>
      <rect x="20" y="124" width="40" height="24" rx="4" fill="#8a97a3"></rect>
      <text x="230" y="141" font-size="12" fill="#62707d">row 3</text>
      <text x="20" y="176" font-size="12" fill="#62707d">... one call per row, repeated millions of times</text>
    </g>
    <!-- divider -->
    <line x1="520" y1="20" x2="520" y2="230" stroke="#e2e8ee" stroke-width="1.5"></line>
    <!-- right: photon vectorized -->
    <text x="548" y="28" font-size="14" font-weight="700" fill="#14343f">Photon: a columnar batch at once</text>
    <text x="548" y="46" font-size="12" fill="#62707d">one operation over a whole column of values, in C++</text>
    <rect x="548" y="60" width="460" height="88" rx="6" fill="#f0ecf8" stroke="#8E7CC3"></rect>
    <g fill="#8E7CC3">
      <rect x="562" y="74" width="26" height="60" rx="3"></rect><rect x="596" y="74" width="26" height="60" rx="3"></rect><rect x="630" y="74" width="26" height="60" rx="3"></rect><rect x="664" y="74" width="26" height="60" rx="3"></rect><rect x="698" y="74" width="26" height="60" rx="3"></rect><rect x="732" y="74" width="26" height="60" rx="3"></rect><rect x="766" y="74" width="26" height="60" rx="3"></rect><rect x="800" y="74" width="26" height="60" rx="3"></rect><rect x="834" y="74" width="26" height="60" rx="3"></rect><rect x="868" y="74" width="26" height="60" rx="3"></rect><rect x="902" y="74" width="26" height="60" rx="3"></rect><rect x="936" y="74" width="26" height="60" rx="3"></rect>
    </g>
    <text x="548" y="176" font-size="12" fill="#6b4fa0">one vectorized operation over the whole batch</text>
    <!-- bottom band -->
    <rect x="20" y="200" width="988" height="74" rx="8" fill="#14343f"></rect>
    <text x="36" y="228" style="font-size:20px;fill:#ffffff;font-family:-apple-system,BlinkMacSystemFont,Segoe UI,Roboto,Helvetica,Arial,sans-serif">Same APIs, same optimizer. Photon just executes the operators over columnar batches,</text>
    <text x="36" y="258" style="font-size:20px;fill:#ffffff;font-family:-apple-system,BlinkMacSystemFont,Segoe UI,Roboto,Helvetica,Arial,sans-serif">which is why joins, aggregations, scans, and writes run faster.</text>
  </svg>
</div>

<div class="ds-row">
  <div class="ds-card green">
    <div class="ds-card-title">What it accelerates</div>
    <ul>
      <li><strong>Joins</strong> and <strong>aggregations</strong>.</li>
      <li><strong>Scans</strong> and <strong>writes</strong>, including Delta DML.</li>
      <li>Shuffle, through Photon vectorized shuffle, which retires most manual shuffle tuning.</li>
    </ul>
  </div>
  <div class="ds-card coral">
    <div class="ds-card-title">When it leaves Photon</div>
    <ul>
      <li>Pickled Python <strong>UDFs</strong>, Scala UDFs, and Java UDFs.</li>
      <li><strong>RDD</strong> code paths.</li>
      <li>The operator runs in the JVM engine instead, and every row crosses into the Python worker. Part 8 covers the cost of this in detail.</li>
    </ul>
  </div>
  <div class="ds-card blue">
    <div class="ds-card-title">Verify it engaged</div>
    <ul>
      <li>Query Profile, Execution Details: the <strong>percent of task time in Photon</strong>.</li>
      <li>The Query Profile labels each operator with its engine (Photon or standard JVM).</li>
      <li>The <strong>COVERAGE_PHOTON</strong> insight fires when part of a query left Photon.</li>
    </ul>
  </div>
</div>

</div>

##### EXPAND FOR ADDITIONAL DETAILS

<details>

### What Photon is
- Photon is a vectorized, columnar C++ engine that executes operators in place of the row at a time JVM engine. It is a drop in: the APIs do not change and the optimizer is untouched.
- It is default on for serverless and cannot be turned off there. On classic it is enabled per cluster, and Part 10 covers the DBU trade you make when you turn it on.

### What it accelerates, and what falls back
- Joins, aggregations, scans, and writes run in Photon. Photon vectorized shuffle replaces the classic Tungsten shuffle and cuts memory pressure, which is why most manual shuffle tuning is no longer needed.
- Non vectorizable code leaves Photon for that operator and runs in the JVM engine. Pickled Python UDFs, Scala or Java UDFs, and RDD paths are the common causes. That is the UDF options ranking from Part 8.
- Executor side broadcast moves broadcast builds off the driver, removing a classic bottleneck.

### How to verify it engaged
- Open the Query Profile and look at Execution Details for the percentage of task time spent in Photon. A high percentage means most of the work stayed in the C++ engine.
- In the query plan, the profile labels each operator with its engine. Look for operators running outside Photon to find where the query left it.
- The **COVERAGE_PHOTON** performance insight names a query that left Photon. Same insights family as DATA_SKEW, DATA_SPILL, and EXPLODING_JOIN from Part 2.
- The Query Profile and its Photon metrics are available for SQL warehouse and serverless compute. On classic all-purpose you read Photon engagement from the Spark UI instead.


</details>

## Conclusion

In this part you covered the compute layer:

1. **Four serverless surfaces**: Jobs, Notebooks, DBSQL warehouses, and SDP. No node types, no idle capacity.
2. **Three hard prerequisites**: Standard access mode, Unity Catalog only, Spark Connect compatible code. Enforced at the platform level.
3. **When classic still wins**: Scala, R, GPU, init scripts, private networking. Workspace Base Environments replace dependency init scripts.
4. **Cost and modes**: metered per workload second. Performance optimized for interactive, standard mode for batch.
5. **Photon**: default on serverless, opt in on classic. Verify with percent of task time in Photon and the COVERAGE_PHOTON insight.


&copy; <span id="dbx-year"></span> Databricks, Inc. All rights reserved.
Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>
<script>
  document.getElementById("dbx-year").textContent = new Date().getFullYear();
</script>